# 03 — Robustness Checks

Specification grid: one cell per check type, visual inspection.

Dimensions:
- Treatment: {count, ratio}
- Nuisance learner: {HistGBR, LightGBM, Lasso}
- Sample: {full panel, tenure ≥ 23 weeks}
- Country controls: {with, without}

Additional checks:
- Overlap/positivity
- Outlier exclusion (>3 SD)
- Heckman selection correction
- DoWhy refutation tests

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from src.data_loading import load_processed
from src.features import build_features, CORE_LAG_COLUMNS, get_country_dummies
from src.treatment_outcome import (
    compute_treatment, compute_outcome, demean_within_trader,
    get_analysis_sample,
)
from src.models import naive_ols, fit_linear_dml, fit_causal_forest, prepare_model_arrays
from src.robustness import (
    run_robustness_grid, overlap_check, outlier_exclusion,
    heckman_selection, dowhy_refutation,
)
from src.viz import plot_spec_curve, set_journal_style

set_journal_style()
print('Imports OK')

Imports OK


## 1. Load and Prepare Data

In [2]:
parquet_path = '../data/processed/trader_panel.parquet' if os.path.exists('../data/processed/trader_panel.parquet') else 'data/processed/trader_panel.parquet'
df = load_processed(parquet_path)
df = build_features(df, print_audit=False)
df, _ = compute_treatment(df)
df = compute_outcome(df)

confounder_lag_cols = [f'{c}_l1' for c in CORE_LAG_COLUMNS]
df_analysis, sample_counts = get_analysis_sample(
    df, treatment_col='total_self_trxn',
    confounder_lag_cols=confounder_lag_cols,
    require_tenure=True,
)

if 'user_country_name_bucketed' in df_analysis.columns:
    df_analysis, country_dummy_cols = get_country_dummies(df_analysis)
else:
    country_dummy_cols = []

all_confounder_cols = confounder_lag_cols + ['tenure_weeks']
print(f'Analysis sample: {len(df_analysis):,} rows')

2026-08-19 09:49:40,621 [INFO] src.data_loading: Loaded processed data: 1,222,680 rows from ../data/processed/trader_panel.parquet
2026-08-19 09:49:40,621 [INFO] src.features: Starting feature engineering pipeline ...
2026-08-19 09:49:40,636 [INFO] src.features: Step 1: Computing lagged confounders
2026-08-19 09:49:41,538 [INFO] src.features:   Created 'learner_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,580 [INFO] src.features:   Created 'total_follower_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,615 [INFO] src.features:   Created 'guru_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,647 [INFO] src.features:   Created 'followed_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,695 [INFO] src.features:   Created 'gain_difference_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,813 [INFO] src.features:   Created 'greater_than_ten_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:49:41,858 [INFO] src.feature

Analysis sample: 1,196,100 rows


## 2. Specification Grid

In [3]:
robustness_df = run_robustness_grid(
    df_analysis,
    base_confounder_cols=all_confounder_cols,
    country_dummy_cols=country_dummy_cols,
)
print(f'\nCompleted {len(robustness_df)} specifications')
display(robustness_df)

2026-08-19 09:49:44,500 [INFO] src.robustness: Running robustness grid: 24 specifications
2026-08-19 09:49:44,519 [INFO] src.robustness:   Spec 1/24: T=count, nuisance=histgbr, sample=full, country=with_country
2026-08-19 09:49:48,041 [INFO] src.models: Fitting LinearDML: nuisance=histgbr, cv=5, N=1,196,100, W_cols=23
2026-08-19 09:50:28,607 [INFO] src.models: LinearDML ATE: -0.5980 (SE=0.0514, 95% CI=[-0.6987, -0.4974], p=2.418e-31)
2026-08-19 09:50:28,607 [INFO] src.robustness:   Spec 2/24: T=count, nuisance=histgbr, sample=full, country=without_country
2026-08-19 09:50:29,232 [INFO] src.models: Fitting LinearDML: nuisance=histgbr, cv=5, N=1,196,100, W_cols=8
2026-08-19 09:50:54,238 [INFO] src.models: LinearDML ATE: -0.5877 (SE=0.0514, 95% CI=[-0.6884, -0.4870], p=2.769e-30)
2026-08-19 09:50:54,238 [INFO] src.robustness:   Spec 3/24: T=count, nuisance=histgbr, sample=tenure_ge_23, country=with_country
2026-08-19 09:50:55,096 [INFO] src.models: Fitting LinearDML: nuisance=histgbr, cv=


Completed 24 specifications


,spec_id,treatment_type,nuisance_learner,sample,country_controls,ate,se,ci_lower,ci_upper,pvalue,n
0,1,count,histgbr,full,with_country,-0.598014,0.051351,-0.698661,-0.497367,2.418093e-31,1196100
1,2,count,histgbr,full,without_country,-0.587690,0.051390,-0.688414,-0.486967,2.769474e-30,1196100
2,3,count,histgbr,tenure_ge_23,with_country,-0.688386,0.055470,-0.797106,-0.579667,2.305156e-35,1066669
3,4,count,histgbr,tenure_ge_23,without_country,-0.686379,0.055552,-0.795259,-0.577499,4.542497e-35,1066669
4,5,count,lightgbm,full,with_country,-0.582437,0.050000,-0.680436,-0.484438,2.331408e-31,1196100
5,6,count,lightgbm,full,without_country,-0.589855,0.050563,-0.688957,-0.490753,1.908592e-31,1196100
6,7,count,lightgbm,tenure_ge_23,with_country,-0.680447,0.054347,-0.786965,-0.573929,5.771558e-36,1066669
7,8,count,lightgbm,tenure_ge_23,without_country,-0.676108,0.054923,-0.783755,-0.568461,7.988379e-35,1066669
8,9,count,lasso,full,with_country,-0.695712,0.047170,-0.788164,-0.603260,3.124695e-49,1196100
9,10,count,lasso,full,without_country,-0.695587,0.047154,-0.788008,-0.603167,3.017129e-49,1196100


In [4]:
# Save results
robustness_df.to_csv('../data/processed/robustness_results.csv', index=False)

# Specification curve
plot_spec_curve(robustness_df, figdir='../figures')

2026-08-19 09:54:32,301 [INFO] src.viz: Saved: ../figures\fig7_spec_curve.png, ../figures\fig7_spec_curve.pdf


'../figures\\fig7_spec_curve.png'

## 3. Overlap/Positivity Check

In [5]:
arrays = prepare_model_arrays(
    df_analysis,
    treatment_col='total_self_trxn', outcome_col='total_gain_w',
    confounder_cols=all_confounder_cols,
)

overlap = overlap_check(
    T=arrays['T'], W=arrays['W'],
    feature_names=all_confounder_cols,
)

print('\n=== OVERLAP CHECK ===')
for name, diag in overlap.items():
    if 'error' not in diag:
        t_range = diag['t_range_across_bins']
        print(f'  {name:30s}: T range across deciles = {t_range:.4f}')
    else:
        print(f'  {name:30s}: ERROR - {diag["error"]}')

2026-08-19 09:54:32,344 [INFO] src.models: Model arrays: Y((1196100,)), T((1196100,)), W((1196100, 8)), X(None)
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\robustness.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("bin")["T"]
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\robustness.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("bin")["T"]
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\robustness.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pas


=== OVERLAP CHECK ===
  learner_count_l1              : T range across deciles = 0.0000
  total_follower_count_l1       : T range across deciles = 0.2993
  guru_count_l1                 : T range across deciles = 0.2055
  followed_count_l1             : T range across deciles = 0.2246
  gain_difference_l1            : T range across deciles = 0.8607
  greater_than_ten_l1           : T range across deciles = 0.0000
  total_trxn_l1                 : T range across deciles = 0.0000
  tenure_weeks                  : T range across deciles = 0.1481


D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\robustness.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("bin")["T"]


## 4. Outlier Exclusion

In [6]:
df_no_outliers, outlier_stats = outlier_exclusion(df_analysis.copy())
print(f'\nOutlier exclusion: {outlier_stats}')

# Re-fit DML on outlier-excluded sample
if len(df_no_outliers) > 1000:
    arr_out = prepare_model_arrays(
        df_no_outliers,
        treatment_col='total_self_trxn', outcome_col='total_gain_w',
        confounder_cols=all_confounder_cols,
    )
    _, dml_no_outliers = fit_linear_dml(
        Y=arr_out['Y'], T=arr_out['T'], W=arr_out['W'],
        nuisance_learner='histgbr',
    )
    print(f"DML (no outliers): ATE={dml_no_outliers['ate']:.4f}, CI=[{dml_no_outliers['ate_ci_lower']:.4f}, {dml_no_outliers['ate_ci_upper']:.4f}]")

2026-08-19 09:54:33,735 [INFO] src.robustness: Outlier exclusion (>3.0 SD): 1,196,100 → 1,161,989 (removed 34,111)
2026-08-19 09:54:33,760 [INFO] src.models: Model arrays: Y((1161989,)), T((1161989,)), W((1161989, 8)), X(None)
2026-08-19 09:54:33,760 [INFO] src.models: Fitting LinearDML: nuisance=histgbr, cv=5, N=1,161,989, W_cols=8



Outlier exclusion: {'original_n': 1196100, 'total_self_trxn_outliers': 6391, 'total_self_trxn_outlier_pct': 0.005343198729203244, 'total_gain_w_outliers': 27720, 'total_gain_w_outlier_pct': 0.023299815332993194, 'final_n': 1161989, 'total_removed': 34111}


2026-08-19 09:54:51,073 [INFO] src.models: LinearDML ATE: 1.3674 (SE=0.1162, 95% CI=[1.1396, 1.5952], p=5.93e-32)


DML (no outliers): ATE=1.3674, CI=[1.1396, 1.5952]


## 5. Heckman Selection Correction

In [7]:
heckman = heckman_selection(df, df_analysis)
print('\n=== HECKMAN SELECTION ===')
for k, v in heckman.items():
    print(f'  {k}: {v}')


=== HECKMAN SELECTION ===
  probit_converged: True
  probit_pseudo_r2: 0.015586422886748497
  imr_mean: 1.911865969721888
  imr_std: 0.18661796529535932
  selection_rate: 0.06646546929695424
  n_full: 1222680
  n_active: 81266


## 6. DoWhy Refutation Tests

In [8]:
dowhy_results = dowhy_refutation(
    df_analysis,
    treatment_col='total_self_trxn',
    outcome_col='total_gain_w',
    confounder_cols=all_confounder_cols,
)

print('\n=== DOWHY REFUTATION ===')
for k, v in dowhy_results.items():
    print(f'  {k}: {v}')

2026-08-19 09:54:56,810 [WARNING] dowhy.causal_model: Causal Graph not provided. DoWhy will construct a graph based on data inputs.
2026-08-19 09:54:56,811 [INFO] dowhy.causal_model: Model to find the causal effect of treatment ['total_self_trxn'] on outcome ['total_gain_w']
2026-08-19 09:54:56,812 [WARNING] dowhy.causal_model: There are an additional 61 variables in the dataset that are not in the graph. Variable names are: '['active_followed_trader_count', 'country_Canada', 'country_France', 'country_Germany', 'country_Hungary', 'country_India', 'country_Italy', 'country_Malaysia', 'country_Norway', 'country_Other', 'country_Portugal', 'country_Russia', 'country_South Africa', 'country_Spain', 'country_Switzerland', 'country_United Kingdom', 'followed_count', 'follower_country_count', 'foreign_country_follower_count', 'foreign_country_learner_count', 'gain_difference', 'greater_than_ten', 'guru_count', 'joining_date', 'keep_country', 'learner_count', 'log_active_followed_trader_count


=== DOWHY REFUTATION ===
  subsampled: True
  n: 50000
  baseline_estimate: -0.46900729663294904
  placebo_effect: -0.01145157992943177
  placebo_pvalue: 0.3943522026064217
  random_cause_effect: -0.4690514645284065
  subset_effect: -0.4556662972037958


## 7. Summary Table

In [9]:
if len(robustness_df) > 0:
    print('\n=== ROBUSTNESS SUMMARY ===')
    valid = robustness_df.dropna(subset=['ate'])
    print(f'  Specifications run:     {len(valid)}')
    print(f'  ATE range:              [{valid["ate"].min():.4f}, {valid["ate"].max():.4f}]')
    print(f'  Median ATE:             {valid["ate"].median():.4f}')
    print(f'  All same sign:          {(valid["ate"] > 0).all() or (valid["ate"] < 0).all()}')
    n_sig = (valid['pvalue'] < 0.05).sum()
    print(f'  Significant (p<0.05):   {n_sig}/{len(valid)}')


=== ROBUSTNESS SUMMARY ===
  Specifications run:     24
  ATE range:              [-22.7537, -0.5824]
  Median ATE:             -10.2217
  All same sign:          True
  Significant (p<0.05):   24/24
